In [5]:
!nvidia-smi

Sun May 17 17:01:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |     169MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# 1. 设置超参数
batch_size = 128
learning_rate = 0.001
epochs = 10


# 2. 设置设备
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"

print("Using device:", device)
print("Using AMP:", use_amp)


# 3. 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


# 4. 下载 MNIST 数据集
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


# 5. 创建 DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)


# 6. 定义 3 层 MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)


model = MLP().to(device)


# 7. 定义损失函数、优化器和 AMP Scaler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


# 8. 训练函数
def train_one_epoch(epoch):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(
        f"Epoch [{epoch}/{epochs}] "
        f"Train Loss: {avg_loss:.4f}, "
        f"Train Acc: {accuracy:.4f}"
    )


# 9. 测试函数
def evaluate(epoch):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    print(
        f"Epoch [{epoch}/{epochs}] "
        f"Test  Loss: {avg_loss:.4f}, "
        f"Test  Acc: {accuracy:.4f}"
    )


# 10. 主训练循环
for epoch in range(1, epochs + 1):
    train_one_epoch(epoch)
    evaluate(epoch)


Using device: cuda
Using AMP: True


/tmp/ipykernel_1895/2470869697.py:87: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_1895/2470869697.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [1/10] Train Loss: 0.2745, Train Acc: 0.9193


/tmp/ipykernel_1895/2470869697.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch [1/10] Test  Loss: 0.1245, Test  Acc: 0.9610
Epoch [2/10] Train Loss: 0.1009, Train Acc: 0.9694
Epoch [2/10] Test  Loss: 0.0892, Test  Acc: 0.9731
Epoch [3/10] Train Loss: 0.0685, Train Acc: 0.9785
Epoch [3/10] Test  Loss: 0.0866, Test  Acc: 0.9727
Epoch [4/10] Train Loss: 0.0501, Train Acc: 0.9843
Epoch [4/10] Test  Loss: 0.0899, Test  Acc: 0.9714
Epoch [5/10] Train Loss: 0.0381, Train Acc: 0.9878
Epoch [5/10] Test  Loss: 0.0724, Test  Acc: 0.9772
Epoch [6/10] Train Loss: 0.0332, Train Acc: 0.9891
Epoch [6/10] Test  Loss: 0.0764, Test  Acc: 0.9776
Epoch [7/10] Train Loss: 0.0218, Train Acc: 0.9927
Epoch [7/10] Test  Loss: 0.0781, Test  Acc: 0.9772
Epoch [8/10] Train Loss: 0.0215, Train Acc: 0.9928
Epoch [8/10] Test  Loss: 0.0809, Test  Acc: 0.9774
Epoch [9/10] Train Loss: 0.0216, Train Acc: 0.9925
Epoch [9/10] Test  Loss: 0.0865, Test  Acc: 0.9765
Epoch [10/10] Train Loss: 0.0164, Train Acc: 0.9944
Epoch [10/10] Test  Loss: 0.0865, Test  Acc: 0.9779
